In [1]:
!pip install gradio reportlab requests black autopep8

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 24.2 MB/s eta 0:00:00


In [2]:
import ast

class DocGenieAnalyzer:

    @staticmethod
    def extract_function_signature(code):
        tree = ast.parse(code)

        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):

                params = []

                for arg in node.args.args:
                    param_type = None
                    if arg.annotation:
                        param_type = ast.unparse(arg.annotation)

                    params.append({
                        "name": arg.arg,
                        "type": param_type
                    })

                return_type = None
                if node.returns:
                    return_type = ast.unparse(node.returns)

                return {
                    "name": node.name,
                    "params": params,
                    "return_type": return_type
                }

        return None

In [3]:
class DocGenieLogicAnalyzer:

    @staticmethod
    def analyze_function_logic(code):
        tree = ast.parse(code)

        analysis = {
            "has_loops": False,
            "has_conditions": False,
            "operations": [],
            "has_return": False
        }

        for node in ast.walk(tree):

            if isinstance(node, (ast.For, ast.While)):
                analysis["has_loops"] = True

            if isinstance(node, ast.If):
                analysis["has_conditions"] = True

            if isinstance(node, ast.BinOp):
                analysis["operations"].append(type(node.op).__name__)

            if isinstance(node, ast.Return):
                analysis["has_return"] = True

        return analysis

In [4]:
class DocGenieFormatter:

    @staticmethod
    def generate_google_docstring(signature, analysis):

        doc = f'"""{signature["name"].replace("_", " ").title()}.\n\n'

        if analysis["has_conditions"]:
            doc += "Contains conditional logic.\n"

        if analysis["has_loops"]:
            doc += "Uses loops in execution.\n"

        doc += "\nArgs:\n"

        for param in signature["params"]:
            doc += f'    {param["name"]} ({param["type"] or "Any"}): Description.\n'

        doc += "\nReturns:\n"
        doc += f'    {signature["return_type"] or "Any"}: Result of function.\n'

        doc += '"""'

        return doc

In [6]:
    @staticmethod
    def generate_numpy_docstring(signature, analysis):

        doc = f'"""{signature["name"].replace("_", " ").title()}.\n\n'

        doc += "Parameters\n----------\n"

        for param in signature["params"]:
            doc += f'{param["name"]} : {param["type"] or "Any"}\n'
            doc += "    Description.\n"

        doc += "\nReturns\n-------\n"
        doc += f'{signature["return_type"] or "Any"}\n'
        doc += "    Result of function.\n"

        doc += '\n"""'

        return doc

In [7]:
import gradio as gr

def generate_doc(code, style):

    signature = DocGenieAnalyzer.extract_function_signature(code)

    if not signature:
        return "No function found."

    analysis = DocGenieLogicAnalyzer.analyze_function_logic(code)

    if style == "google":
        docstring = DocGenieFormatter.generate_google_docstring(signature, analysis)
    else:
        docstring = DocGenieFormatter.generate_numpy_docstring(signature, analysis)

    lines = code.split("\n")

    for i, line in enumerate(lines):
        if line.strip().startswith("def "):
            lines.insert(i+1, "    " + docstring)
            break

    return "\n".join(lines)


with gr.Blocks() as app:

    gr.Markdown("# 🧠 Doc-Genie")

    code_input = gr.Code(
        value="def factorial(n: int) -> int:\n    if n == 0:\n        return 1\n    return n * factorial(n-1)",
        language="python",
        label="Enter Python Function"
    )

    style = gr.Radio(["google", "numpy"], value="google", label="Docstring Style")

    output = gr.Code(language="python", label="Generated Code")

    btn = gr.Button("Generate Docstring")

    btn.click(generate_doc, inputs=[code_input, style], outputs=output)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f59cf5fe84112666f1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
def calculate(x: int, y: int) -> int:
    if x > y:
        return x - y
    return x + y